# Predicting House Prices in King County with Linear Regression

### **Please keep in mind that this sample case focuses on the model comparison and ensemble methods. The exploratory data analysis, pre-processing, and analysis of important features (coefficients and informative attributes) are omitted for brevity. This case should not be used as a reference for the final project. The methods covered here can by combined with the content in case 4.1 for a more complete analysis.**

## Problem Statement
---------------------------

The problem on hand is to predict the price of a home based on a collection of specific characteristics describing it. In the process, we need to identify the most important features in the dataset. We need to employ techniques of data preprocessing and build a linear regression model that predicts the prices.

<img src="https://static.seattletimes.com/wp-content/uploads/2018/10/111313-780x502.jpg" title="source: imgur.com" />

----------------------------
## Data Information
---------------------------



Attribute Information (in order):

--------------------------------------------
| Variable      | Description                                   |
|---------------|-----------------------------------------------|
| Price         | Prices of the houses (Target Variable)         |
| Bedrooms      | Number of bedrooms                            |
| Bathrooms     | Number of bathrooms                           |
| Floors        | Number of floors                              |
| sqft_livin    | Square footage of the home                    |
| sqft_lot      | Square footage of the lot                     |
| floors        | Total floors (levels) in house                |
| waterfront    | House with a view to a waterfront             |
| view          | Has been viewed                               |
| condition     | Overall condition of the house                |
| grade         | Overall grade given to the housing unit       |
| sqft_above    | Square footage of house apart from basement   |
| sqft_bmnt     | Square footage of the basement                |
| yr_built      | Built year                                    |
| yr_renov      | Year when house was renovated                 |
| zipcode       | Zip code                                      |
| lat           | Latitude coordinate                           |
| long          | Longitude coordinate                          |
| sqft_l15      | Living room area in 2015 (some renovations)   |
| sqft_lt15     | Lot size area in 2015 (some renovations)      |





[Link to detailed variable definitions](https://github.com/shwetapai/Predicting-House-Prices-in-King-County)

### We will start be importing the necessary libraries

In [43]:
# import libraries for data manipulation
import pandas as pd
import numpy as np

# import libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.gofplots import ProbPlot

# import libraries for building regression models
from statsmodels.formula.api import ols
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# import libraries for model evaluation
from sklearn import metrics
from sklearn.model_selection import cross_val_score

# import library for preparing data
from sklearn.model_selection import train_test_split

# import library for data preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

### Read in the data

In [44]:
import requests
import pandas as pd
from io import StringIO

orig_url="https://drive.google.com/file/d/1A9a9SwoUwVc9C91H4U62Z3RmwmMvjx7x/view?usp=drive_link"

file_id = orig_url.split('/')[-2]
dwn_url='https://drive.google.com/uc?export=download&id=' + file_id
url = requests.get(dwn_url).text
csv_raw = StringIO(url)
df = pd.read_csv(csv_raw, sep=",")
df.head()


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


Drop the date variable for now -- it will be problematic in numerical computations.

In [45]:
# @title
df=df.drop(['date','id', 'lat','long'], axis=1) # axis=0 means drop a row, axis=1 means drop a column

### Transformation of Target Variable
The target variable is slightly skewed. In such cases it is a good idead to apply a log transformation in order to avoid contradicting the assumption of linearity which is a necessary condition for applying a linear regression model.

In [46]:
# @title
df['price_log'] = np.log(df['price'])

---------------------------
## Preprocessing the Data
---------------------------


### Separate the target variable from the collection of feature variables.

In [47]:
# @title
# separate the dependent and indepedent variable
log_y = df['price_log']
y=df['price']
X = df.drop(columns = ['price', 'price_log'])
# X = df.drop(['price', 'price_log'], axis=1) # alternative code

### Checking for Multicollinearity with VIF

In [48]:
# @title
from statsmodels.stats.outliers_influence import variance_inflation_factor

# function to check VIF
def checking_vif(X):
    vif = pd.DataFrame()
    vif["feature"] = X.columns

    # calculating VIF for each feature
    vif["VIF"] = [
        variance_inflation_factor(X.values, i) for i in range(len(X.columns))
    ]
    return vif

print(checking_vif(X))

          feature          VIF
0        bedrooms    23.230620
1       bathrooms    28.588800
2     sqft_living          inf
3        sqft_lot     2.365789
4          floors    16.694998
5      waterfront     1.212246
6            view     1.537038
7       condition    34.591655
8           grade   140.426251
9      sqft_above          inf
10  sqft_basement          inf
11       yr_built  8789.263294
12   yr_renovated     1.192089
13        zipcode  8591.842472
14  sqft_living15    26.514005
15     sqft_lot15     2.582123


### Drop variables with high VIF (*do not drop the constant*)

**Drop the columns labeled:**
- 'sqft_living'
- 'sqft_above'
- 'sqft_basement'

from the training data and testing data and check if multicollinearity is removed

### Drop the columns indicated above from the training and testing sets.

In [49]:
# @title
# create the model after dropping variables with high VIF
X = X.drop(columns=['sqft_living','sqft_above','sqft_basement'])

# check for VIF
print(checking_vif(X))

          feature          VIF
0        bedrooms    20.505602
1       bathrooms    22.932267
2        sqft_lot     2.351259
3          floors    13.751455
4      waterfront     1.209246
5            view     1.481000
6       condition    34.473396
7           grade   120.235573
8        yr_built  8469.541030
9    yr_renovated     1.191904
10        zipcode  8372.792932
11  sqft_living15    21.438717
12     sqft_lot15     2.570192


### Scale the Feature Variables
**Feature scaling** is important for both **Linear Regression** and **Support Vector Regression (SVR)** because these models are sensitive to the scale of the input data.

1. **Linear Regression**:
   - In linear regression, coefficients are calculated based on the values of each feature. If features are on different scales (e.g., one feature in thousands and another between 0 and 1), the model’s coefficients can become skewed, leading to **biased predictions**.
   - Scaling ensures that all features contribute equally, making the model more interpretable and improving numerical stability.

2. **Support Vector Regression (SVR)**:
   - SVR relies on **distance-based calculations** to find the optimal margin (epsilon) and support vectors. If features are on different scales, those with larger values will dominate the distance calculations, causing the model to be biased toward those features.
   - Scaling the data allows SVR to consider all features fairly, improving its ability to capture relevant patterns in the data.

Scaling helps both models perform better by ensuring that each feature contributes proportionally, making the training process more accurate and efficient.

In [50]:
# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Check the below linear regression assumptions (omitted from this notebook for parsimony)

- **Mean of Residuals**: Should be close to 0 to indicate unbiased predictions.
- **No Heteroscedasticity**: Residuals should have constant variance across different levels of the predictors.
- **Linearity of Variables**: The relationship between predictors and the target variable should be linear, which can be checked by examining residual plots.
- **Normality of Error Terms**: Residuals should be normally distributed, which can be verified using Q-Q plots and histograms.

By checking these assumptions, we ensure that the linear regression model is reliable and can produce valid predictions.

### Checking the Performance of the Model Across Training and Testing Data
Here we create several helper functions that we can use to check the performance metrics of the linear regression model.

- **MSE** (Mean Squared Error):
- **RAE** (Mean Absolute Error):
- **RMSE** (Root Mean Squared Error):
- **R-squared** (Percent of Variation Explained by the Model):



In [51]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_regression_model(model, X_train, X_test, y_train, y_test):
    """
    Evaluates the performance of a regression model on training and testing data.

    Parameters:
    - model: The regression model to evaluate.
    - X_train: Training feature data.
    - X_test: Testing feature data.
    - y_train: Training target data.
    - y_test: Testing target data.

    Returns:
    - DataFrame with performance metrics (MSE, MAE, RMSE, and R-squared) for train and test sets.
    """

    # Predictions on training data
    y_train_pred = model.predict(X_train)

    # Predictions on testing data
    y_test_pred = model.predict(X_test)

    # Calculate metrics for both train and test data
    metrics = {
        "Data": ["Train", "Test"],
        "MSE": [
            mean_squared_error(y_train, y_train_pred),
            mean_squared_error(y_test, y_test_pred),
        ],
        "MAE": [
            mean_absolute_error(y_train, y_train_pred),
            mean_absolute_error(y_test, y_test_pred),
        ],
        "RMSE": [
            np.sqrt(mean_squared_error(y_train, y_train_pred)),
            np.sqrt(mean_squared_error(y_test, y_test_pred)),
        ],
        "R-squared": [
            r2_score(y_train, y_train_pred),
            r2_score(y_test, y_test_pred),
        ],
    }

    # Create a DataFrame to display the metrics
    performance_df = pd.DataFrame(metrics)

    # Print the performance DataFrame
    return performance_df

### Exercise 1.a.
Model A has a Root Mean Squared Error of RMSE of 3.5, while Model B has an RMSE of 2.8. Which model is performing better, and why?   
Model B is performing better. RMSE (Root Mean Squared Error) is an error metric, so a lower score is better. This means Model B's predictions are, on average, closer to the actual house prices (an error of 2.8) than Model A's (an error of 3.5).

---   

### Exercise 1.b.
How does the presence of outliers in the data affect the Mean Squared Error MSE, and why might this make MSE a useful or less useful metric?   
Outliers dramatically increase the MSE. The metric squares the error of each prediction, so a single large error from an outlier (e.g., predicting a USD 5M mansion at UDS 2M) will be exponentially punished.

This makes MSE less useful if you want to know the model's "typical" performance, as one outlier can skew the entire metric and make a good model look bad.

It is more useful in scenarios (like finance) where large errors are catastrophic and you want to heavily penalize models that produce them.

---
      
What metric could be used in place of the MSE that would be less impacted by outliers?    
MAE (Mean Absolute Error). MAE measures the average absolute difference between predictions and actual values, so it doesn't square the errors. This means outliers still contribute to the error, but they do so linearly, not exponentially, giving a much more stable and "typical" measure of model performance.



### Exercise 1.c.   
What does an R-squared (R²) value of 0.85 mean in terms of the model's performance?    
An R-squared of 0.85 means that the model's features (like sqft_living, grade, etc.) can explain 85% of the variability in the house prices. This is generally considered a strong performance, as it indicates a high goodness-of-fit. The remaining 15% of the price variation is due to other factors (randomness or features not included in the model).


### Exercise 1.d.
Can R-squared be negative? If so, what does a negative R-squared value indicate?     
Yes, R-squared can be negative. $R^2$ is a measure of how much better your model is than a dumb baseline model that simply predicts the average price for every single house. A negative $R^2$ value means your model is performing even worse than that simple baseline model and is a sign of an extremely poor model fit.
  


### Linear Regression
---
**Linear Regression** is a foundational algorithm in machine learning for predicting continuous outcomes. It models the relationship between a target variable \( y \) and one or more features \( X \) with a linear equation. The model minimizes the **sum of squared errors** to find the best-fitting line or hyperplane.

### Key Points
- **Simple & Multiple Linear Regression**: Models with one feature or multiple features.
- **Regularization**: Ridge (L2) and Lasso (L1) help prevent overfitting by penalizing large coefficients.
- **Assumptions**: Assumes linearity, independence of errors, constant error variance (homoscedasticity), and normally distributed errors.
- **Evaluation Metrics**: Common metrics include Mean Absolute Error (MAE), Mean Squared Error (MSE), and R-squared (R²).

### Advantages
- Easy to implement and interpret.
- Efficient for large datasets.

### Limitations
- Assumes linear relationships.
- Sensitive to outliers and multicollinearity.

Linear regression is widely used for tasks like sales forecasting and price prediction and serves as a baseline model in machine learning.

### Split the data into training and testing sets
We'll use the scaled data for the linear regression model and the support vector regression model.

In [52]:
#splitting the data in 70:30 ratio of train to test data
X_train_scaled, X_test_scaled, y_train_log, y_test_log = train_test_split(X_scaled, log_y, test_size=0.30 , random_state=42)

### Run the Linear Regression Model

In [53]:
# Create a regression tree model
linear_model = LinearRegression()

# Train the model on the training data
linear_model.fit(X_train_scaled, y_train_log)

LinearRegression()

Run the `evaluate_regression_model()` function on the `linear_model`.

In [54]:
# Checking model performance
evaluate_regression_model(linear_model, X_train_scaled, X_test_scaled, y_train_log, y_test_log)

,Data,MSE,MAE,RMSE,R-squared
0,Train,0.098444,0.249703,0.313758,0.642310
1,Test,0.100406,0.251912,0.316869,0.644412


Because we converted the values of the target variable to a log scale, we can't easily interpret the performance metrics. Below is a process and sample interpretation that can be utilized in these scenarios.

In [55]:
import numpy as np

# Exponentiate RMSE values
train_rmse_original = np.exp(0.314945)
test_rmse_original = np.exp(0.314225)

print("Train RMSE (Original Scale):", train_rmse_original)
print("Test RMSE (Original Scale):", test_rmse_original)


Train RMSE (Original Scale): 1.3701839487673733
Test RMSE (Original Scale): 1.369197771390719


#### Interpretation of back-transformed RMSE
The exponentiated RMSE (1.37) suggests that the model’s predictions are typically off by around 37% from the actual price.

**Interpretation in Terms of House Price Range**:
If, for example, the actual house price is `$300,000`, the model’s prediction might typically fall within a range of:   

$300,000 / 1.37 \approx 219,000 \text{ to } 300,000 \times 1.37 \approx 411,000$


### Observations
   
- The model performance is decent but not exceptional as indicated by the R-squared value which indicates that about 64% of the variation in house price (log-transformed) can be explained by the linear regression model.
   
- The performance metrics for the training and testing sets are very similar. We can therefore conclude that the model is not overfitting and can reasonably be applied to new data without the risk of misrepresenting the results.

- The performance of the linear regression model could be improved with the application of ridge/lasso regression. The alpha value could be tuned through grid search or random search.

- We can also try different models to see if they are better able to capture patterns in the data. Because there are multiple different features in the data set that may not have a linear relationship with house prices, tree models may prove to be effective.

## Examination of Additional Models
Goal: Improve model generalization performance
- Support Vector Regression
- Decision Tree for Regression
- Random Forest Regression
- Hyperparameter Tuned Random Forest (optional)

### Support Vector Regression
---
**Support Vector Regression (SVR)** is a type of Support Vector Machine (SVM) used for predicting continuous outcomes. SVR aims to fit a line (or hyperplane in higher dimensions) within a margin of tolerance, called epsilon, where points inside this margin are considered "close enough" and don’t influence the model. Only points outside the margin, called **support vectors**, affect the model, making it robust to outliers.

### Key Points
- **Epsilon-Insensitive Margin**: Defines a margin where errors are not penalized, allowing the model to ignore minor deviations.
- **Support Vectors**: Only data points outside the epsilon margin affect the model, which helps with generalization.
- **Kernel Trick**: SVR can use kernels (e.g., linear, RBF, polynomial) to map data into higher dimensions, enabling it to capture non-linear relationships.

### Advantages
- **Robust to Outliers**: Only support vectors impact the model, making it less sensitive to outliers.
- **Flexible with Kernels**: Effective for both linear and non-linear data.

### Limitations
- **Slow with Large Datasets**: SVR can be computationally intensive for large datasets.
- **Sensitive to Feature Scaling**: Requires standardized features for optimal performance.

SVR is commonly used for tasks like stock price prediction and trend analysis, especially when small margins of error are acceptable and robustness to outliers is needed.

### Exercise 2.a. Run the Support Vector Regression Model
- Use `SVR()` to instantiate the Support Vector Regression algorithm.   
- Set `kernel='linear'`
- Set `C=1`
- Use `max_iter=10000`

In [56]:
# Create and train the SVR model with adjusted parameters
svr_model = SVR(kernel='linear', C=1, max_iter=10000)
svr_model.fit(X_train_scaled, y_train_log)

SVR(C=1, kernel='linear', max_iter=10000)

Run the `evaluate_regression_model()` function on the `svr_model`.

In [57]:
# Checking model performance
evaluate_regression_model(svr_model, X_train_scaled, X_test_scaled, y_train_log, y_test_log)

,Data,MSE,MAE,RMSE,R-squared
0,Train,0.179635,0.340264,0.423834,0.347310
1,Test,0.186222,0.345212,0.431535,0.340496


In [58]:
import numpy as np

# Exponentiate RMSE values
train_rmse_original = np.exp(0.423834)
test_rmse_original = np.exp(0.431535)

print("Train RMSE (Original Scale):", train_rmse_original)
print("Test RMSE (Original Scale):", test_rmse_original)

Train RMSE (Original Scale): 1.5278079566119567
Test RMSE (Original Scale): 1.539619025836248


#### Interpretation of back-transformed RMSE
The exponentiated RMSE (1.54) suggests that the model’s predictions are typically off by around 54% from the actual price.

**Interpretation in Terms of House Price Range**:
If, for example, the actual house price is `$300,000`, the model’s prediction might typically fall within a range of:   

$300,000 / 1.54 \approx 194,805 \text{ to } 300,000 \times 1.54 \approx 462,000$


### Exercise 2.b. Write some observations about the SVR model performance.
  
The model is stable and not overfitting. The "Train RMSE" (1.528) and "Test RMSE" (1.540) are extremely close, which means the model performs just as well on unseen data as it did on the data it was trained on. This indicates it's a reliable model.

The SVR model's interpretation of error is incorrect. Just like with the previous linear regression model, exponentiating the RMSE (which is an error of logs) to get 1.54 and then claiming it's a "54% error" is not a correct or meaningful statistical interpretation.

The SVR model is performing worse than the simple linear regression model. The SVR's test RMSE on the log-transformed data is 0.4315, which is significantly higher than the linear regression's test RMSE of 0.3142. A higher error means the SVR model's predictions are, on average, further away from the true house prices.



### Decision Tree for Regression
----
**Regression Trees** are a type of decision tree used to predict continuous target values. They work by recursively splitting the data into subsets based on feature values, aiming to minimize prediction error within each split. Unlike linear regression, regression trees can capture complex, non-linear relationships between features and the target.

### Key Points
- **Splitting**: At each node, the algorithm chooses the feature and threshold that best split the data, typically by minimizing the **mean squared error** within each subset.
- **Recursive Structure**: The tree splits each subset until a stopping criterion is met (e.g., maximum depth or minimum samples per leaf).
- **Prediction**: The prediction for any data point is the average of target values in the leaf node it falls into.

### Advantages
- **Handles Non-linear Relationships**: Effective for complex data with non-linear patterns.
- **Interpretability**: Easy to visualize and interpret the decision process.

### Limitations
- **Overfitting**: Regression trees can easily overfit; pruning or setting depth limits helps control this.
- **Sensitivity to Small Changes**: Small changes in data can lead to different splits, making trees unstable.

Regression trees are widely used for tasks like predicting customer spending and real estate prices, especially when relationships are non-linear or hierarchical.

### Split the data into training and testing sets
For tree-based models (no scaling or log transformations). These training and testing sets can be used for the remaining models

In [59]:
#splitting the data in 70:30 ratio of train to test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30 , random_state=42)

### Exercise 3.a. Run the Decision Tree Model

Please complete the code to run a Decision Tree Regressor with `DecisionTreeRegressor()`. Trees do not require scaling or log transformation so we can use the price, `y_train`, `y_test`, as the target variable and the unscaled version of the feature variables, `X_train`,`X_test`.
   
Run this model without any hyperparameter settings and note the model behavior.

In [60]:
# Create a regression tree model
tree_model = DecisionTreeRegressor(random_state=42)

# Train the model on the training data
tree_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

Run the `evaluate_regression_model()` function on the `tree_model`.

In [61]:
# Checking model performance
evaluate_regression_model(tree_model, X_train, X_test, y_train, y_test)

,Data,MSE,MAE,RMSE,R-squared
0,Train,8.664781e+07,884.050124,9308.480363,0.999337
1,Test,6.254551e+10,127095.888315,250091.011461,0.566759


### Exercise 3.b. Write some observations about the tree model performance.
  
The model is severely overfit. This is the most critical observation.

We know it's overfit because its performance on the training data is nearly perfect (R-squared of 0.999), meaning it has "memorized" the data.

However, its performance on the unseen test data is poor, with the R-squared dropping to only 0.567.


### Random Forest Regression
---
**Random Forest Regression** is an ensemble learning method that combines multiple decision trees to improve predictive accuracy and stability. In a random forest, each tree is trained on a random subset of data and features, and the final prediction is the average of all tree predictions. This approach reduces overfitting compared to a single tree and makes the model more robust.

### Key Points
- **Bootstrap Aggregation (Bagging)**: Each tree is trained on a different random subset of data (with replacement), helping to reduce variance.
- **Random Feature Selection**: At each split, only a random subset of features is considered, further decorrelating the trees and improving generalization.
- **Prediction**: The model outputs the average of predictions from all trees, smoothing out individual tree errors.

### Advantages
- **Reduced Overfitting**: By averaging multiple trees, random forests are less prone to overfitting than individual trees.
- **High Accuracy**: Effective on complex data with non-linear relationships.
- **Feature Importance**: Provides insights into which features are most influential.

### Limitations
- **Interpretability**: Harder to interpret than single trees.
- **Computationally Intensive**: Slower and more resource-intensive for large datasets due to multiple tree training.

Random forests are widely used for tasks like house price prediction and risk assessment, particularly when high accuracy and stability are required.

### Exercise 4.a. Run the Random Forest Model

Please complete the code to run a Random Forest Regressor with `RandomForestRegressor()`. Trees do not require scaling or log transformation so we can use the price, `y_train`, `y_test`, as the target variable and the unscaled version of the feature variables, `X_train`,`X_test`.
   
Use `n_estimators=100` as the number of tree models to run on the randomly seledcted subsets of the data.

In [62]:
from sklearn.ensemble import RandomForestRegressor

# Create a Random Forest Regression model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)  # You can adjust hyperparameters as needed

# Train the model on the training data
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

Run the `evaluate_regression_model()` function on the `rf_model`.

In [63]:
# Checking model performance
evaluate_regression_model(rf_model, X_train, X_test, y_train, y_test)

,Data,MSE,MAE,RMSE,R-squared
0,Train,3.679217e+09,33508.494253,60656.552351,0.971838
1,Test,3.275605e+10,93208.228264,180986.334743,0.773105


### Exercise 4.b. Write some observations about the Random Forest model performance
The model shows signs of overfitting. Its performance on the training data is nearly perfect (R-squared of 0.972), but its R-squared drops to 0.773 on the test set. The same pattern is seen in the RMSE, which triples from $60,656 on the train set to $180,986 on the test set.

Despite overfitting, this model is far superior to the single Decision Tree model (which had a test R-squared of 0.567). The Random Forest's ensemble method (bagging) has significantly improved its ability to generalize to new data, explaining 77.3% of the price variation.

The model's Mean Absolute Error (MAE) on the test set is USD 93,208. This is a clear, interpretable metric suggesting that, on average, the model's price prediction is off by about $93,000.
### Observations on Random Forest Model Performance



### Gradient Boosting Regressor

**Gradient Boosting Regression** is an ensemble method that builds a sequence of decision trees, each correcting errors from the previous one. Unlike random forests, which average predictions from multiple trees, gradient boosting combines trees sequentially to minimize residual errors, making it highly effective for complex regression tasks.

### Key Points
- **Sequential Learning**: Each new tree is trained on the residuals (errors) of the previous trees, focusing on the harder-to-predict instances.
- **Learning Rate**: A parameter that controls the contribution of each tree to the final model, balancing accuracy and overfitting.
- **Additive Model**: The final prediction is the sum of predictions from all trees, each improving the accuracy incrementally.

### Advantages
- **High Accuracy**: Gradient boosting often achieves superior predictive performance by refining predictions iteratively.
- **Handles Non-Linear Relationships**: Effective for data with complex patterns and interactions.

### Limitations
- **Risk of Overfitting**: Requires careful tuning of parameters (e.g., learning rate, tree depth) to prevent overfitting.
- **Computationally Intensive**: Training can be slow, especially for large datasets.

Gradient boosting is widely used in applications requiring high accuracy, such as sales forecasting and financial modeling, where capturing complex relationships in the data is essential.



### Gradient Boosting Regressor with varried learning rate

Train a Gradient Boosting Regressor on the dataset and tune the learning_rate parameter (e.g., try values of 0.01, 0.1, and 0.2). Record the test MSE for each learning rate and analyze how increasing the learning rate impacts model performance and the risk of overfitting.

In [64]:
from sklearn.ensemble import GradientBoostingRegressor

# Initialize model with different learning rates
for lr in [0.01, 0.1, 0.2, 0.5]:
    gb_model = GradientBoostingRegressor(learning_rate=lr)
    gb_model.fit(X_train, y_train)

    # Evaluate on test set
    y_test_pred_gb = gb_model.predict(X_test)
    mse_gb = mean_squared_error(y_test, y_test_pred_gb)
    print(f"Learning Rate: {lr}, Test MSE: {mse_gb}")

Learning Rate: 0.01, Test MSE: 72053273504.1402
Learning Rate: 0.1, Test MSE: 33976861383.696087
Learning Rate: 0.2, Test MSE: 29919415028.05158
Learning Rate: 0.5, Test MSE: 24864421976.210896


### Exercise 5.a. What is the nature of the relationship between the learning rate and the Mean Squared Error?   
Based on the output provided, there is an inverse relationship in this specific range: as the learning_rate increases from 0.01 to 0.5, the Test MSE (error) consistently decreases.

This indicates that a very low learning rate (like 0.01) causes the model to learn too slowly and underfit, resulting in a high error (7.2e10). Increasing the rate to 0.5 allows the model to learn more "aggressively" with each new tree, leading to a better fit and the lowest error in this experiment (2.49e10).

However, this doesn't mean a higher learning rate is always better. This relationship is typically U-shaped: if the rate becomes too high, the model will "overshoot" the optimal solution, and the error will begin to increase again.
### Observations on Learning Rate and Mean Squared Error

As the **learning rate increases**, the **Mean Squared Error (MSE) decreases** on the test set. This suggests an **inverse relationship** between the learning rate and MSE in this context:

- **Lower learning rates** (e.g., 0.01) result in higher MSE, likely because the model takes smaller steps toward minimizing errors, which can lead to underfitting as the model may not be fully optimized.
- **Higher learning rates** (e.g., 0.5) result in a lower MSE, indicating that the model is more effective at capturing patterns in the data as it takes larger steps during optimization.

However, increasing the learning rate too much can risk **overfitting** if the model starts to fit the training data too closely. Finding a balance is crucial, as very high learning rates may reduce MSE initially but could harm performance on new data due to overfitting.

### Evaluation
Run the `evaluate_regression_model()` function on the gb_model.    
**----**


In [65]:
evaluate_regression_model(gb_model, X_train, X_test, y_train, y_test)

,Data,MSE,MAE,RMSE,R-squared
0,Train,1.152368e+10,72672.055792,107348.414733,0.911795
1,Test,2.486442e+10,87703.816986,157684.564800,0.827769


### Exercise 5.b. Write some observations about the Gradient Boosting Regressor.   
This model is the best performer so far, achieving a Test R-squared of 0.827 (82.7%). This is a significant improvement over the Random Forest (0.773) and the single Decision Tree (0.567), explaining nearly 83% of the price variability.

The model is still overfitting, as shown by the performance gap between the Train R-squared (0.912) and the Test R-squared (0.827). However, its ability to generalize to new data is much stronger than the other tree models.

The Test MAE of USD 87,823 is the lowest error we've seen, beating the Random Forest's USD 93,208. This means the Gradient Boosting model's predictions are, on average, about $5,400 more accurate than the Random Forest's.

### Exercise 6 - Overall Conclusions and Suggestions for Improvement
Please write an overall summary of this regression model comparison and implementation of ensemble methods. Consider:
- Which is the best performing model and what does this suggest about the data?

Gradient Boosting Regressor was the best performing model, achieving the highest R-squared (0.827) and lowest MAE ($87,823) on the test set. The fact that all the tree-based ensembles (Random Forest and Gradient Boosting) dramatically outperformed the linear models (Linear Regression, SVR, Decision Tree) strongly suggests that the relationship between house features and price is highly non-linear and complex. Price is not driven by simple, straight-line relationships but by complex interactions between features (e.g., the value of sqft_living likely depends on the zipcode).


- What additional techniques could be used to further improve model performance?

The model's performance could be improved by hyperparameter tuning; we could use GridSearchCV on the Gradient Boosting model's learning_rate, n_estimators, and max_depth to reduce its overfitting and boost the test score. We could also perform feature engineering by creating new, more insightful features like Age_of_House (Current_Year - yr_built) or Years_Since_Renovation.


- How could the selected model be implemented in a business setting to provide actionable insights and provide a strategic advantage to stakeholders?

The Gradient Boosting model could be deployed as a dynamic pricing tool for real estate agents or a public-facing website to provide instant, data-driven price estimates (like a "Zestimate"). Stakeholders could also use it to run "what-if" scenarios (e.g., "How much does the predicted price increase if we add a bathroom and increase the grade?") to identify high-ROI renovations and undervalued properties, providing a clear strategic advantage in investment or house-flipping.


### Hyperparamter Tuning Random Forest Model (optional)
Hyperparameter tuning in a **Random Forest Regressor** involves adjusting parameters that control the model’s complexity, accuracy, and robustness. Commonly tuned hyperparameters include:

1. **Number of Trees (`n_estimators`)**: Controls the number of decision trees in the forest. More trees usually improve accuracy but increase computation time.

2. **Maximum Tree Depth (`max_depth`)**: Limits the depth of each tree, controlling overfitting. Shallower trees generalize better but may underfit if too shallow.

3. **Minimum Samples per Split (`min_samples_split`)**: Sets the minimum number of samples required to split a node. Higher values can reduce overfitting by preventing overly fine splits.

4. **Minimum Samples per Leaf (`min_samples_leaf`)**: Sets the minimum number of samples in each leaf node. Larger values help prevent overly complex trees.

5. **Maximum Features (`max_features`)**: Determines the number of features considered when splitting nodes. Lower values reduce variance and computation time, but too few can lead to underfitting.

6. **Bootstrap (`bootstrap`)**: Controls whether sampling with replacement is used. Enabling it (True) generally increases robustness.

### Tuning Methods
- **Grid Search**: Tests all combinations of specified parameter values (e.g., using `GridSearchCV`), which can be time-consuming but thorough.
- **Randomized Search**: Tests a random subset of parameter combinations, allowing faster tuning over a broad parameter space.

These methods optimize for metrics like Mean Squared Error (MSE) on cross-validated splits, identifying the best hyperparameter values for model performance.

In [ ]:
from sklearn.model_selection import GridSearchCV

# # Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# # Create a Random Forest Regression model
rf_model = RandomForestRegressor(random_state=42)

# # Create a GridSearchCV object
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2)

# # Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# # Print the best parameters found by the grid search
print("Best parameters found: ", grid_search.best_params_)

# # Get the best model
best_rf_model = grid_search.best_estimator_

# # Evaluate the best model on the test data
y_train_pred_best_rf = best_rf_model.predict(X_train)
y_test_pred_best_rf = best_rf_model.predict(X_test)
print("R-squared for the best Random Forest model (train): ", r2_score(y_train, y_train_pred_best_rf))
print("R-squared for the best Random Forest model (test): ", r2_score(y_test, y_test_pred_best_rf))

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found:  {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
R-squared for the best Random Forest model (train):  0.97263454264826
R-squared for the best Random Forest model (test):  0.7727604288182761
